---
id: "readme"
template: "markdown"
name: "readme"
position:
  x: 0
  y: -40
dimensions:
  width: 900
  height: 360
config:
  md: |-
    # m_SALES_BY_CUSTOMER_MONTH (converted from Informatica PowerCenter)

    | Informatica | Lakeflow Designer |
    |---|---|
    | SQ_ORDERS (Source Filter `STATUS = 'COMPLETED'`) | `src_orders` + `sq_orders_filter` |
    | SQ_ORDER_LINES | `src_order_lines` |
    | JNR_ORDER_LINES (Normal Join, master = ORDERS) | `jnr_order_lines` (inner) |
    | EXP_LINE_AMOUNT (v_PRICE variable port inlined) | `exp_line_amount` |
    | AGG_CUSTOMER_MONTH | `agg_customer_month` (Aggregate) + `agg_round` (ROUND outside the aggregate) |
    | T_SALES_CUSTOMER_MONTH | `out_sales_customer_month` |
---

In [0]:
"""
id: "src_orders"
template: "source"
templateVersion: "1.0.0"
name: "src_orders"
position:
  x: 0
  y: 360
description:
  text: "SQ_ORDERS source."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_orders"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_orders.data"] = out["data"]


In [0]:
"""
id: "sq_orders_filter"
template: "filter"
templateVersion: "1.0.0"
name: "sq_orders_filter"
position:
  x: 300
  y: 360
description:
  text: "SQ_ORDERS Source Filter: ORDERS.STATUS = 'COMPLETED'."
  hash: ""
previewMode: "1000"
config:
  condition: "status = 'COMPLETED'"
input:
  - node: "src_orders"
    input_port: "data"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "status = 'COMPLETED'"
}
inputs = {
    "data": ctx["src_orders.data"]
}
out = run(config, inputs, spark)
ctx["sq_orders_filter.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "src_order_lines"
template: "source"
templateVersion: "1.0.0"
name: "src_order_lines"
position:
  x: 0
  y: 515
description:
  text: "SQ_ORDER_LINES source."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_order_lines"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_order_lines"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_order_lines.data"] = out["data"]


In [0]:
"""
id: "jnr_order_lines"
template: "join"
templateVersion: "1.0.0"
name: "jnr_order_lines"
position:
  x: 600
  y: 360
description:
  text: "JNR_ORDER_LINES: Normal Join on ORDER_ID = ORDER_ID1."
  hash: ""
previewMode: "1000"
config:
  join_type: "inner"
  join_conditions: "left.`order_id` = right.`order_id`"
  expressions:
    - "left.`order_id` AS `order_id`"
    - "left.`customer_id` AS `customer_id`"
    - "left.`order_date` AS `order_date`"
    - "right.`line_no` AS `line_no`"
    - "right.`quantity` AS `quantity`"
    - "right.`unit_price` AS `unit_price`"
    - "right.`discount_pct` AS `discount_pct`"
input:
  - node: "sq_orders_filter"
    input_port: "left"
    output_port: "filtered_data"
  - node: "src_order_lines"
    input_port: "right"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "join_conditions": "left.`order_id` = right.`order_id`",
    "expressions": [
        "left.`order_id` AS `order_id`",
        "left.`customer_id` AS `customer_id`",
        "left.`order_date` AS `order_date`",
        "right.`line_no` AS `line_no`",
        "right.`quantity` AS `quantity`",
        "right.`unit_price` AS `unit_price`",
        "right.`discount_pct` AS `discount_pct`"
    ]
}
inputs = {
    "left": ctx["sq_orders_filter.filtered_data"],
    "right": ctx["src_order_lines.data"]
}
out = run(config, inputs, spark)
ctx["jnr_order_lines.joined_data"] = out["joined_data"]


In [0]:
"""
id: "exp_line_amount"
template: "transform"
templateVersion: "1.0.0"
name: "exp_line_amount"
position:
  x: 900
  y: 360
description:
  text: "EXP_LINE_AMOUNT: v_PRICE = IIF(ISNULL(UNIT_PRICE),0,UNIT_PRICE); TO_CHAR(ORDER_DATE,'YYYY-MM')."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "quantity * COALESCE(unit_price, 0) * (1 - discount_pct) AS line_amount"
    - "DATE_FORMAT(order_date, 'yyyy-MM') AS order_month"
input:
  - node: "jnr_order_lines"
    input_port: "data"
    output_port: "joined_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "quantity * COALESCE(unit_price, 0) * (1 - discount_pct) AS line_amount",
        "DATE_FORMAT(order_date, 'yyyy-MM') AS order_month"
    ]
}
inputs = {
    "data": ctx["jnr_order_lines.joined_data"]
}
out = run(config, inputs, spark)
ctx["exp_line_amount.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "agg_customer_month"
template: "aggregate"
templateVersion: "1.0.0"
name: "agg_customer_month"
position:
  x: 1200
  y: 360
description:
  text: "AGG_CUSTOMER_MONTH: group by CUSTOMER_ID, ORDER_MONTH."
  hash: ""
previewMode: "1000"
config:
  group_bys:
    - expr: "customer_id"
      type: "expr"
    - expr: "order_month"
      type: "expr"
  aggregations:
    - columnExpr:
        expr: "line_amount"
        type: "expr"
      fn: "SUM"
      alias: "total_amount_raw"
    - columnExpr:
        expr: "quantity"
        type: "expr"
      fn: "SUM"
      alias: "total_qty"
    - columnExpr:
        expr: "line_no"
        type: "expr"
      fn: "COUNT"
      alias: "line_count"
    - columnExpr:
        expr: "order_date"
        type: "expr"
      fn: "MAX"
      alias: "last_order_date"
input:
  - node: "exp_line_amount"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

# Default percentile (median). Used as a fallback when a PERCENTILE aggregation
# is missing or has a non-numeric / non-finite percentage. Mirrors
# DEFAULT_PERCENTILE_VALUE in AggregateNodeViewModel.ts so SQL and Python
# pipelines agree on the default.
DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        # Skip pass-through columns that duplicate a group-by column;
        # groupBy() already includes them in the output.
        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.col(raw_expr)
        elif fn == "PERCENTILE":
            # Spark PERCENTILE requires (col, percentage [, frequency]) with percentage
            # in [0, 1]; emitting PERCENTILE(col) triggers WRONG_NUM_ARGS at analysis
            # time. For finite numeric inputs we clamp to [0, 1] (mirroring the V1/V2
            # input clamps) so out-of-range values like -0.1 or 1.5 — typos or stale
            # frontend state — become a valid boundary instead of silently overriding
            # the user's intent with the median. Non-numeric, non-finite, boolean, or
            # missing values fall back to DEFAULT_PERCENTILE.
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "customer_id",
            "type": "expr"
        },
        {
            "expr": "order_month",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "line_amount",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "total_amount_raw"
        },
        {
            "columnExpr": {
                "expr": "quantity",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "total_qty"
        },
        {
            "columnExpr": {
                "expr": "line_no",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "line_count"
        },
        {
            "columnExpr": {
                "expr": "order_date",
                "type": "expr"
            },
            "fn": "MAX",
            "alias": "last_order_date"
        }
    ]
}
inputs = {
    "data": ctx["exp_line_amount.transformed_data"]
}
out = run(config, inputs, spark)
ctx["agg_customer_month.aggregated_data"] = out["aggregated_data"]


In [0]:
"""
id: "agg_round"
template: "transform"
templateVersion: "1.0.0"
name: "agg_round"
position:
  x: 1500
  y: 360
description:
  text: "TOTAL_AMOUNT = ROUND(SUM(o_LINE_AMOUNT), 2): rounding applied after the aggregate."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_id"
    - "order_month"
    - "ROUND(total_amount_raw, 2) AS total_amount"
    - "total_qty"
    - "line_count"
    - "last_order_date"
input:
  - node: "agg_customer_month"
    input_port: "data"
    output_port: "aggregated_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_id",
        "order_month",
        "ROUND(total_amount_raw, 2) AS total_amount",
        "total_qty",
        "line_count",
        "last_order_date"
    ]
}
inputs = {
    "data": ctx["agg_customer_month.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["agg_round.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "out_sales_customer_month"
template: "output"
templateVersion: "1.0.0"
name: "out_sales_customer_month"
position:
  x: 1800
  y: 360
description:
  text: "T_SALES_CUSTOMER_MONTH (truncate + load)."
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "infa_02_sales_customer_month"
input:
  - node: "agg_round"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "infa_02_sales_customer_month"
}
inputs = {
    "data": ctx["agg_round.transformed_data"]
}
out = run(config, inputs, spark)
